In [1]:
# ============================================================
# MICRO STEP 1: Load and inspect training-ready data
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------
BASE = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

path_terrain_train = f"{BASE}\\severn_terrain_ready.parquet"
path_terrain_test  = f"{BASE}\\northumbria_terrain_ready.parquet"
path_era5          = f"{BASE}\\era5_features.parquet"

# ------------------------------------------------------------
# 2. Load files
# ------------------------------------------------------------
print("Loading Severn terrain (training) ...")
df_train = pd.read_parquet(path_terrain_train)

print("Loading Northumbria terrain (test) ...")
df_test = pd.read_parquet(path_terrain_test)

print("Loading ERA5 features ...")
df_era5 = pd.read_parquet(path_era5)

print("All files loaded.\n")

# ------------------------------------------------------------
# 3. Basic shapes and dtypes
# ------------------------------------------------------------
print("=" * 60)
print("SHAPES")
print("=" * 60)
print(f"Severn (train):    {df_train.shape[0]:>12,} rows x {df_train.shape[1]} cols")
print(f"Northumbria (test):{df_test.shape[0]:>12,} rows x {df_test.shape[1]} cols")
print(f"ERA5 features:     {df_era5.shape[0]:>12,} rows x {df_era5.shape[1]} cols")

print("\n" + "=" * 60)
print("TERRAIN COLUMNS & DTYPES (Severn)")
print("=" * 60)
print(df_train.dtypes.to_string())

print("\n" + "=" * 60)
print("ERA5 COLUMNS (first 20)")
print("=" * 60)
print(df_era5.columns.tolist()[:20])
print(f"... ({df_era5.shape[1]} total)")

# ------------------------------------------------------------
# 4. Null check
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("NULL COUNTS")
print("=" * 60)
print("Severn nulls:")
print(df_train.isnull().sum().to_string())
print(f"\nNorthumbria nulls:")
print(df_test.isnull().sum().to_string())
print(f"\nERA5 nulls: {df_era5.isnull().sum().sum()}")

# ------------------------------------------------------------
# 5. Class distributions for all five risk targets
# ------------------------------------------------------------
risk_cols = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]

print("\n" + "=" * 60)
print("CLASS DISTRIBUTIONS -- SEVERN (TRAIN)")
print("=" * 60)
for col in risk_cols:
    counts = df_train[col].value_counts().sort_index()
    pcts = (counts / len(df_train) * 100).round(2)
    print(f"\n{col}:")
    for cls_val in counts.index:
        print(f"  Class {cls_val}: {counts[cls_val]:>12,}  ({pcts[cls_val]:>6.2f}%)")

print("\n" + "=" * 60)
print("CLASS DISTRIBUTIONS -- NORTHUMBRIA (TEST)")
print("=" * 60)
for col in risk_cols:
    counts = df_test[col].value_counts().sort_index()
    pcts = (counts / len(df_test) * 100).round(2)
    print(f"\n{col}:")
    for cls_val in counts.index:
        print(f"  Class {cls_val}: {counts[cls_val]:>12,}  ({pcts[cls_val]:>6.2f}%)")

# ------------------------------------------------------------
# 6. Feature summary stats (terrain)
# ------------------------------------------------------------
feature_cols = ["dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"]

print("\n" + "=" * 60)
print("FEATURE SUMMARY -- SEVERN (TRAIN)")
print("=" * 60)
print(df_train[feature_cols].describe().round(4).to_string())

print("\n" + "=" * 60)
print("FEATURE SUMMARY -- NORTHUMBRIA (TEST)")
print("=" * 60)
print(df_test[feature_cols].describe().round(4).to_string())

# ------------------------------------------------------------
# 7. ERA5 features peek
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("ERA5 FEATURES -- BOTH REGIONS")
print("=" * 60)
print(df_era5.to_string())

# ------------------------------------------------------------
# 8. Unique clc_type values
# ------------------------------------------------------------
train_clc = set(df_train["clc_type"].unique())
test_clc  = set(df_test["clc_type"].unique())
print("\n" + "=" * 60)
print("CLC_TYPE COVERAGE")
print("=" * 60)
print(f"Severn unique classes:     {sorted(train_clc)}")
print(f"Northumbria unique classes:{sorted(test_clc)}")
print(f"In Severn but not Northumbria: {sorted(train_clc - test_clc)}")
print(f"In Northumbria but not Severn: {sorted(test_clc - train_clc)}")

print("\nMicro Step 1 complete. Ready for Step 2.")

Loading Severn terrain (training) ...
Loading Northumbria terrain (test) ...
Loading ERA5 features ...
All files loaded.

SHAPES
Severn (train):      35,278,130 rows x 13 cols
Northumbria (test):  21,391,824 rows x 13 cols
ERA5 features:                2 rows x 103 cols

TERRAIN COLUMNS & DTYPES (Severn)
proj_y          float32
proj_x          float32
dtm_zscore      float32
log_flow_acc    float32
imd             float32
waw                int8
clc_type          int16
risk_0_2m          int8
risk_0_3m          int8
risk_0_6m          int8
risk_0_9m          int8
risk_1_2m          int8
is_waterway        int8

ERA5 COLUMNS (first 20)
['region', 'max_rolling_5d_tp_mean', 'max_rolling_10d_tp_mean', 'max_rolling_15d_tp_mean', 'max_rolling_5d_tp_max', 'max_rolling_10d_tp_max', 'max_rolling_15d_tp_max', 'max_rolling_5d_sro_mean', 'max_rolling_10d_sro_mean', 'max_rolling_15d_sro_mean', 'max_rolling_5d_sro_max', 'max_rolling_10d_sro_max', 'max_rolling_15d_sro_max', 'max_rolling_7d_swvl1_mean

In [2]:
# ============================================================
# MICRO STEP 2: Stratified subsample for development speed
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load training data
# ------------------------------------------------------------
BASE = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"
df_train = pd.read_parquet(f"{BASE}\\severn_terrain_ready.parquet")

# ------------------------------------------------------------
# 2. Stratified sample: 2M rows, stratified on risk_0_2m
#    This preserves class proportions while being fast to train on.
# ------------------------------------------------------------
TARGET = "risk_0_2m"
SAMPLE_SIZE = 2_000_000
RANDOM_SEED = 42

# Compute how many rows per class to maintain proportions
class_counts = df_train[TARGET].value_counts()
class_fracs = class_counts / len(df_train)
samples_per_class = (class_fracs * SAMPLE_SIZE).round().astype(int)

# Adjust rounding so total equals exactly SAMPLE_SIZE
diff = SAMPLE_SIZE - samples_per_class.sum()
samples_per_class.iloc[0] += diff  # add/subtract remainder to largest class

print("Target samples per class:")
for cls_val in sorted(samples_per_class.index):
    print(f"  Class {cls_val}: {samples_per_class[cls_val]:>10,}")
print(f"  Total:   {samples_per_class.sum():>10,}")

# ------------------------------------------------------------
# 3. Draw the stratified sample
# ------------------------------------------------------------
rng = np.random.default_rng(RANDOM_SEED)
sampled_parts = []

for cls_val in sorted(samples_per_class.index):
    cls_mask = df_train[TARGET] == cls_val
    cls_indices = df_train.index[cls_mask]
    chosen = rng.choice(cls_indices, size=samples_per_class[cls_val], replace=False)
    sampled_parts.append(df_train.loc[chosen])

df_sample = pd.concat(sampled_parts, ignore_index=True)

# Shuffle so classes are not in contiguous blocks
df_sample = df_sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nSample shape: {df_sample.shape}")

# ------------------------------------------------------------
# 4. Verify class distribution is preserved
# ------------------------------------------------------------
print("\nClass distribution comparison:")
print(f"{'Class':<8} {'Full %':>10} {'Sample %':>10} {'Sample N':>12}")
print("-" * 44)
for cls_val in sorted(class_counts.index):
    full_pct = class_counts[cls_val] / len(df_train) * 100
    sample_ct = (df_sample[TARGET] == cls_val).sum()
    sample_pct = sample_ct / len(df_sample) * 100
    print(f"{cls_val:<8} {full_pct:>9.2f}% {sample_pct:>9.2f}% {sample_ct:>12,}")

# ------------------------------------------------------------
# 5. Save the subsample for reuse
# ------------------------------------------------------------
out_path = f"{BASE}\\severn_sample_2M.parquet"
df_sample.to_parquet(out_path, index=False)
print(f"\nSaved to: {out_path}")
print("Micro Step 2 complete. Ready for Step 3.")

Target samples per class:
  Class 0:  1,811,348
  Class 1:     48,028
  Class 2:     43,144
  Class 3:     24,681
  Class 4:     72,799
  Total:    2,000,000

Sample shape: (2000000, 13)

Class distribution comparison:
Class        Full %   Sample %     Sample N
--------------------------------------------
0            90.57%     90.57%    1,811,348
1             2.40%      2.40%       48,028
2             2.16%      2.16%       43,144
3             1.23%      1.23%       24,681
4             3.64%      3.64%       72,799

Saved to: C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\severn_sample_2M.parquet
Micro Step 2 complete. Ready for Step 3.


In [3]:
# ============================================================
# MICRO STEP 3: Define features, target, and preprocessing
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load the subsample and ERA5 features
# ------------------------------------------------------------
BASE = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

df = pd.read_parquet(f"{BASE}\\severn_sample_2M.parquet")
df_era5 = pd.read_parquet(f"{BASE}\\era5_features.parquet")

print(f"Sample shape: {df.shape}")
print(f"ERA5 shape:   {df_era5.shape}")

# ------------------------------------------------------------
# 2. Extract Severn ERA5 features and broadcast onto every row
# ------------------------------------------------------------
era5_severn = df_era5[df_era5["region"] == "severn"].drop(columns=["region"])
print(f"\nERA5 feature count: {era5_severn.shape[1]}")

# Broadcast: assign each ERA5 value as a constant column
for col in era5_severn.columns:
    df[col] = era5_severn[col].values[0]

print(f"Shape after ERA5 broadcast: {df.shape}")

# ------------------------------------------------------------
# 3. Define target and feature columns
# ------------------------------------------------------------
TARGET = "risk_0_2m"

# Columns that are NOT features
coord_cols  = ["proj_y", "proj_x"]
risk_cols   = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]
exclude     = set(coord_cols + risk_cols)

# Everything else is a feature
feature_cols = [c for c in df.columns if c not in exclude]

# Separate terrain-only features (for the baseline in step 4)
terrain_feature_cols = [
    "dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"
]
era5_feature_cols = [c for c in feature_cols if c not in terrain_feature_cols]

print(f"\nTerrain features:  {len(terrain_feature_cols)}")
print(f"ERA5 features:     {len(era5_feature_cols)}")
print(f"Total features:    {len(feature_cols)}")
print(f"Target:            {TARGET}")

# ------------------------------------------------------------
# 4. Prepare X and y
# ------------------------------------------------------------
X = df[feature_cols].copy()
y = df[TARGET].copy()

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y dtype: {y.dtype}")
print(f"y unique values: {sorted(y.unique())}")

# ------------------------------------------------------------
# 5. Mark clc_type as categorical for LightGBM
# ------------------------------------------------------------
X["clc_type"] = X["clc_type"].astype("category")
print(f"\nclc_type dtype after conversion: {X['clc_type'].dtype}")
print(f"clc_type categories: {X['clc_type'].cat.categories.tolist()}")

# ------------------------------------------------------------
# 6. Train/validation split (80/20, stratified)
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nX_train: {X_train.shape}  |  X_val: {X_val.shape}")
print(f"y_train distribution:")
for cls_val in sorted(y_train.unique()):
    ct = (y_train == cls_val).sum()
    print(f"  Class {cls_val}: {ct:>10,}  ({ct/len(y_train)*100:.2f}%)")

# ------------------------------------------------------------
# 7. Compute class weights for imbalance handling
# ------------------------------------------------------------
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(y_train.unique()))
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print("\nComputed class weights (balanced):")
for cls_val, w in class_weight_dict.items():
    print(f"  Class {cls_val}: {w:.4f}")

# ------------------------------------------------------------
# 8. Build sample_weight array for LightGBM
# ------------------------------------------------------------
sample_weight_train = y_train.map(class_weight_dict).values
print(f"\nSample weight array shape: {sample_weight_train.shape}")
print(f"Sample weight range: [{sample_weight_train.min():.4f}, {sample_weight_train.max():.4f}]")

# ------------------------------------------------------------
# 9. Save split indices for reproducibility
# ------------------------------------------------------------
split_info = {
    "feature_cols": feature_cols,
    "terrain_feature_cols": terrain_feature_cols,
    "era5_feature_cols": era5_feature_cols,
    "target": TARGET,
    "class_weight_dict": class_weight_dict,
    "train_size": len(X_train),
    "val_size": len(X_val),
}

import json

# Save feature lists as JSON for reference
info_path = f"{BASE}\\model_config.json"
saveable = {k: v for k, v in split_info.items()}
saveable["class_weight_dict"] = {str(k): v for k, v in class_weight_dict.items()}
with open(info_path, "w") as f:
    json.dump(saveable, f, indent=2)

print(f"\nConfig saved to: {info_path}")
print("Micro Step 3 complete. Ready for Step 4 (baseline LightGBM).")

Sample shape: (2000000, 13)
ERA5 shape:   (2, 103)

ERA5 feature count: 102
Shape after ERA5 broadcast: (2000000, 115)

Terrain features:  6
ERA5 features:     102
Total features:    108
Target:            risk_0_2m


C:\Users\jackp\AppData\Local\Temp\ipykernel_11164\2385584573.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = era5_severn[col].values[0]
C:\Users\jackp\AppData\Local\Temp\ipykernel_11164\2385584573.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = era5_severn[col].values[0]
C:\Users\jackp\AppData\Local\Temp\ipykernel_11164\2385584573.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 


X shape: (2000000, 108)
y shape: (2000000,)
y dtype: int8
y unique values: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]

clc_type dtype after conversion: category
clc_type categories: [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142, 211, 222, 231, 242, 243, 311, 312, 313, 321, 322, 324, 333, 411, 412, 421, 423, 511, 512, 522, 523]

X_train: (1600000, 108)  |  X_val: (400000, 108)
y_train distribution:
  Class 0:  1,449,078  (90.57%)
  Class 1:     38,423  (2.40%)
  Class 2:     34,515  (2.16%)
  Class 3:     19,745  (1.23%)
  Class 4:     58,239  (3.64%)

Computed class weights (balanced):
  Class 0: 0.2208
  Class 1: 8.3283
  Class 2: 9.2713
  Class 3: 16.2066
  Class 4: 5.4946

Sample weight array shape: (1600000,)
Sample weight range: [0.2208, 16.2066]

Config saved to: C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\model_config.json
Micro Step 3 complete. Ready for Step 4 (baseline LightGBM).


In [4]:
# ============================================================
# MICRO STEP 4: Baseline LightGBM (terrain features only)
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
import time

# ------------------------------------------------------------
# 1. Load and prepare data (same as step 3)
# ------------------------------------------------------------
BASE = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

df = pd.read_parquet(f"{BASE}\\severn_sample_2M.parquet")

TARGET = "risk_0_2m"
terrain_feature_cols = [
    "dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"
]

X = df[terrain_feature_cols].copy()
y = df[TARGET].copy()

X["clc_type"] = X["clc_type"].astype("category")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class weights
classes = np.array(sorted(y_train.unique()))
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
sample_weight_train = y_train.map(class_weight_dict).values
sample_weight_val   = y_val.map(class_weight_dict).values

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

# ------------------------------------------------------------
# 2. Create LightGBM datasets
# ------------------------------------------------------------
dtrain = lgb.Dataset(
    X_train, label=y_train, weight=sample_weight_train,
    categorical_feature=["clc_type"], free_raw_data=False
)
dval = lgb.Dataset(
    X_val, label=y_val, weight=sample_weight_val,
    reference=dtrain, free_raw_data=False
)

# ------------------------------------------------------------
# 3. Define baseline parameters
# ------------------------------------------------------------
params = {
    "objective": "multiclass",
    "num_class": 5,
    "metric": "multi_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "seed": 42,
    "n_jobs": -1,
}

# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------
print("\nTraining baseline (terrain only) ...")
t0 = time.time()

callbacks = [
    lgb.log_evaluation(period=50),
    lgb.early_stopping(stopping_rounds=50, verbose=True),
]

model_baseline = lgb.train(
    params,
    dtrain,
    num_boost_round=500,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=callbacks,
)

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Best iteration: {model_baseline.best_iteration}")

# ------------------------------------------------------------
# 5. Predict and evaluate on validation set
# ------------------------------------------------------------
y_pred_proba = model_baseline.predict(X_val, num_iteration=model_baseline.best_iteration)
y_pred = np.argmax(y_pred_proba, axis=1)

acc = accuracy_score(y_val, y_pred)
f1_macro = f1_score(y_val, y_pred, average="macro")
f1_weighted = f1_score(y_val, y_pred, average="weighted")

print("\n" + "=" * 60)
print("BASELINE RESULTS (terrain only, validation set)")
print("=" * 60)
print(f"Accuracy:         {acc:.4f}")
print(f"Macro F1:         {f1_macro:.4f}")
print(f"Weighted F1:      {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(y_val, y_pred)
print(cm)

# ------------------------------------------------------------
# 6. Feature importance
# ------------------------------------------------------------
importance = model_baseline.feature_importance(importance_type="gain")
feat_imp = pd.DataFrame({
    "feature": terrain_feature_cols,
    "importance": importance
}).sort_values("importance", ascending=False)

print("\nFeature Importance (gain):")
for _, row in feat_imp.iterrows():
    bar = "#" * int(row["importance"] / feat_imp["importance"].max() * 40)
    print(f"  {row['feature']:<16} {row['importance']:>12,.0f}  {bar}")

# ------------------------------------------------------------
# 7. Save baseline model
# ------------------------------------------------------------
model_path = f"{BASE}\\baseline_terrain_lgb.txt"
model_baseline.save_model(model_path)
print(f"\nModel saved to: {model_path}")
print("Micro Step 4 complete. Ready for Step 5 (add ERA5 features).")

X_train: (1600000, 6)  |  X_val: (400000, 6)

Training baseline (terrain only) ...
Training until validation scores don't improve for 50 rounds
[50]	train's multi_logloss: 1.31867	val's multi_logloss: 1.3297
[100]	train's multi_logloss: 1.29515	val's multi_logloss: 1.31653
[150]	train's multi_logloss: 1.28244	val's multi_logloss: 1.31395
[200]	train's multi_logloss: 1.27208	val's multi_logloss: 1.31283
[250]	train's multi_logloss: 1.26344	val's multi_logloss: 1.31272
[300]	train's multi_logloss: 1.25525	val's multi_logloss: 1.31282
Early stopping, best iteration is:
[282]	train's multi_logloss: 1.25804	val's multi_logloss: 1.31257

Training complete in 51.1s
Best iteration: 282

BASELINE RESULTS (terrain only, validation set)
Accuracy:         0.6670
Macro F1:         0.3058
Weighted F1:      0.7574

Classification Report:
              precision    recall  f1-score   support

           0     0.9760    0.6963    0.8128    362270
           1     0.1021    0.3254    0.1555      9605
  

In [5]:
# ============================================================
# MICRO STEP 5: Add ERA5 weather features and retrain
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import time

# ------------------------------------------------------------
# 1. Load data and broadcast ERA5
# ------------------------------------------------------------
BASE = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

df = pd.read_parquet(f"{BASE}\\severn_sample_2M.parquet")
df_era5 = pd.read_parquet(f"{BASE}\\era5_features.parquet")

era5_severn = df_era5[df_era5["region"] == "severn"].drop(columns=["region"])

# Broadcast ERA5 efficiently (avoid fragmentation warning)
era5_broadcast = pd.DataFrame(
    np.repeat(era5_severn.values, len(df), axis=0),
    columns=era5_severn.columns,
    index=df.index,
)
df = pd.concat([df, era5_broadcast], axis=1)

print(f"Shape after ERA5 broadcast: {df.shape}")

# ------------------------------------------------------------
# 2. Define features and target
# ------------------------------------------------------------
TARGET = "risk_0_2m"
coord_cols = ["proj_y", "proj_x"]
risk_cols = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]
exclude = set(coord_cols + risk_cols)

all_feature_cols = [c for c in df.columns if c not in exclude]
terrain_feature_cols = [
    "dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"
]
era5_feature_cols = [c for c in all_feature_cols if c not in terrain_feature_cols]

print(f"Total features: {len(all_feature_cols)} (6 terrain + {len(era5_feature_cols)} ERA5)")

X = df[all_feature_cols].copy()
y = df[TARGET].copy()
X["clc_type"] = X["clc_type"].astype("category")

# ------------------------------------------------------------
# 3. Train/val split (same seed as baseline for fair comparison)
# ------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

classes = np.array(sorted(y_train.unique()))
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
sample_weight_train = y_train.map(class_weight_dict).values
sample_weight_val   = y_val.map(class_weight_dict).values

# ------------------------------------------------------------
# 4. LightGBM datasets
# ------------------------------------------------------------
dtrain = lgb.Dataset(
    X_train, label=y_train, weight=sample_weight_train,
    categorical_feature=["clc_type"], free_raw_data=False
)
dval = lgb.Dataset(
    X_val, label=y_val, weight=sample_weight_val,
    reference=dtrain, free_raw_data=False
)

# ------------------------------------------------------------
# 5. Same params as baseline (fair comparison)
# ------------------------------------------------------------
params = {
    "objective": "multiclass",
    "num_class": 5,
    "metric": "multi_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "seed": 42,
    "n_jobs": -1,
}

# ------------------------------------------------------------
# 6. Train
# ------------------------------------------------------------
print("\nTraining with terrain + ERA5 features ...")
t0 = time.time()

callbacks = [
    lgb.log_evaluation(period=50),
    lgb.early_stopping(stopping_rounds=50, verbose=True),
]

model_full = lgb.train(
    params,
    dtrain,
    num_boost_round=500,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=callbacks,
)

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Best iteration: {model_full.best_iteration}")

# ------------------------------------------------------------
# 7. Evaluate
# ------------------------------------------------------------
y_pred_proba = model_full.predict(X_val, num_iteration=model_full.best_iteration)
y_pred = np.argmax(y_pred_proba, axis=1)

acc = accuracy_score(y_val, y_pred)
f1_macro = f1_score(y_val, y_pred, average="macro")
f1_weighted = f1_score(y_val, y_pred, average="weighted")

print("\n" + "=" * 60)
print("FULL MODEL RESULTS (terrain + ERA5, validation set)")
print("=" * 60)
print(f"Accuracy:         {acc:.4f}")
print(f"Macro F1:         {f1_macro:.4f}")
print(f"Weighted F1:      {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(y_val, y_pred)
print(cm)

# ------------------------------------------------------------
# 8. Compare against baseline
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("COMPARISON: BASELINE vs FULL MODEL")
print("=" * 60)
print(f"{'Metric':<20} {'Baseline':>10} {'+ ERA5':>10} {'Delta':>10}")
print("-" * 52)
print(f"{'Accuracy':<20} {'0.6670':>10} {acc:>10.4f} {acc - 0.6670:>+10.4f}")
print(f"{'Macro F1':<20} {'0.3058':>10} {f1_macro:>10.4f} {f1_macro - 0.3058:>+10.4f}")
print(f"{'Weighted F1':<20} {'0.7574':>10} {f1_weighted:>10.4f} {f1_weighted - 0.7574:>+10.4f}")

# ------------------------------------------------------------
# 9. Top 20 feature importance
# ------------------------------------------------------------
importance = model_full.feature_importance(importance_type="gain")
feat_imp = pd.DataFrame({
    "feature": all_feature_cols,
    "importance": importance
}).sort_values("importance", ascending=False)

print("\nTop 20 Feature Importance (gain):")
top20 = feat_imp.head(20)
for _, row in top20.iterrows():
    bar = "#" * int(row["importance"] / top20["importance"].max() * 40)
    print(f"  {row['feature']:<35} {row['importance']:>12,.0f}  {bar}")

# ------------------------------------------------------------
# 10. Save model
# ------------------------------------------------------------
model_path = f"{BASE}\\full_terrain_era5_lgb.txt"
model_full.save_model(model_path)
print(f"\nModel saved to: {model_path}")
print("Micro Step 5 complete. Ready for Step 6 (hyperparameter tuning).")

Shape after ERA5 broadcast: (2000000, 115)
Total features: 108 (6 terrain + 102 ERA5)

Training with terrain + ERA5 features ...
Training until validation scores don't improve for 50 rounds
[50]	train's multi_logloss: 1.31867	val's multi_logloss: 1.3297
[100]	train's multi_logloss: 1.29515	val's multi_logloss: 1.31653
[150]	train's multi_logloss: 1.28244	val's multi_logloss: 1.31395
[200]	train's multi_logloss: 1.27208	val's multi_logloss: 1.31283
[250]	train's multi_logloss: 1.26344	val's multi_logloss: 1.31272
[300]	train's multi_logloss: 1.25525	val's multi_logloss: 1.31282
Early stopping, best iteration is:
[282]	train's multi_logloss: 1.25804	val's multi_logloss: 1.31257

Training complete in 55.4s
Best iteration: 282

FULL MODEL RESULTS (terrain + ERA5, validation set)
Accuracy:         0.6670
Macro F1:         0.3058
Weighted F1:      0.7574

Classification Report:
              precision    recall  f1-score   support

           0     0.9760    0.6963    0.8128    362270
      

In [8]:
# ============================================================
# MICRO STEP 6: Hyperparameter tuning (terrain only)
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import optuna
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ------------------------------------------------------------
# 1. Load and prepare data
# ------------------------------------------------------------
BASE = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

df = pd.read_parquet(f"{BASE}\\severn_sample_2M.parquet")

TARGET = "risk_0_2m"
terrain_feature_cols = [
    "dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"
]

X = df[terrain_feature_cols].copy()
y = df[TARGET].copy()
X["clc_type"] = X["clc_type"].astype("category")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

classes = np.array(sorted(y_train.unique()))
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
sample_weight_train = y_train.map(class_weight_dict).values
sample_weight_val   = y_val.map(class_weight_dict).values

dtrain = lgb.Dataset(
    X_train, label=y_train, weight=sample_weight_train,
    categorical_feature=["clc_type"], free_raw_data=False
)
dval = lgb.Dataset(
    X_val, label=y_val, weight=sample_weight_val,
    reference=dtrain, free_raw_data=False
)

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

# ------------------------------------------------------------
# 2. Define Optuna objective
# ------------------------------------------------------------
def objective(trial):
    params = {
        "objective": "multiclass",
        "num_class": 5,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbose": -1,
        "seed": 42,
        "n_jobs": -1,

        # Tuned hyperparameters
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 255),
        "max_depth":         trial.suggest_int("max_depth", 4, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 500, log=True),
        "feature_fraction":  trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq":      trial.suggest_int("bagging_freq", 1, 10),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        valid_names=["val"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=False),
            lgb.log_evaluation(period=0),  # silent
        ],
    )

    y_pred = np.argmax(model.predict(X_val, num_iteration=model.best_iteration), axis=1)
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    return macro_f1

# ------------------------------------------------------------
# 3. Run Optuna search (50 trials)
# ------------------------------------------------------------
print("\nStarting Optuna search (50 trials) ...")
t0 = time.time()

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

elapsed = time.time() - t0
print(f"\nSearch complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")

# ------------------------------------------------------------
# 4. Report best result
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("BEST TRIAL")
print("=" * 60)
print(f"Macro F1: {study.best_value:.4f}  (baseline was 0.3058)")
print(f"Improvement: {study.best_value - 0.3058:+.4f}")

print("\nBest hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k:<22} {v}")

# ------------------------------------------------------------
# 5. Retrain best model and get full evaluation
# ------------------------------------------------------------
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

best_params = {
    "objective": "multiclass",
    "num_class": 5,
    "metric": "multi_logloss",
    "boosting_type": "gbdt",
    "verbose": -1,
    "seed": 42,
    "n_jobs": -1,
    **study.best_params,
}

print("\nRetraining best model ...")
model_tuned = lgb.train(
    best_params,
    dtrain,
    num_boost_round=500,
    valid_sets=[dval],
    valid_names=["val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)

y_pred_proba = model_tuned.predict(X_val, num_iteration=model_tuned.best_iteration)
y_pred = np.argmax(y_pred_proba, axis=1)

acc = accuracy_score(y_val, y_pred)
f1_macro = f1_score(y_val, y_pred, average="macro")
f1_weighted = f1_score(y_val, y_pred, average="weighted")

print("\n" + "=" * 60)
print("TUNED MODEL RESULTS (validation set)")
print("=" * 60)
print(f"Accuracy:         {acc:.4f}")
print(f"Macro F1:         {f1_macro:.4f}")
print(f"Weighted F1:      {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# Feature importance
importance = model_tuned.feature_importance(importance_type="gain")
feat_imp = pd.DataFrame({
    "feature": terrain_feature_cols,
    "importance": importance
}).sort_values("importance", ascending=False)

print("\nFeature Importance (gain):")
for _, row in feat_imp.iterrows():
    bar = "#" * int(row["importance"] / feat_imp["importance"].max() * 40)
    print(f"  {row['feature']:<16} {row['importance']:>12,.0f}  {bar}")

# ------------------------------------------------------------
# 6. Save tuned model
# ------------------------------------------------------------
model_path = f"{BASE}\\tuned_terrain_lgb.txt"
model_tuned.save_model(model_path)
print(f"\nModel saved to: {model_path}")
print("Micro Step 6 complete. Ready for Step 7 (full data training).")

X_train: (1600000, 6)  |  X_val: (400000, 6)

Starting Optuna search (50 trials) ...


  0%|          | 0/50 [00:00<?, ?it/s]


Search complete in 4651.4s (77.5 min)

BEST TRIAL
Macro F1: 0.3089  (baseline was 0.3058)
Improvement: +0.0031

Best hyperparameters:
  learning_rate          0.010088181430903833
  num_leaves             190
  max_depth              9
  min_child_samples      56
  feature_fraction       0.5550624139934653
  bagging_fraction       0.5894363245398411
  bagging_freq           2
  reg_alpha              0.48339848293388055
  reg_lambda             0.003247853729391959

Retraining best model ...
Training until validation scores don't improve for 50 rounds
[100]	val's multi_logloss: 1.42306
[200]	val's multi_logloss: 1.36512
[300]	val's multi_logloss: 1.34143
[400]	val's multi_logloss: 1.32989
[500]	val's multi_logloss: 1.32345
Did not meet early stopping. Best iteration is:
[500]	val's multi_logloss: 1.32345

TUNED MODEL RESULTS (validation set)
Accuracy:         0.6729
Macro F1:         0.3089
Weighted F1:      0.7622

Classification Report:
              precision    recall  f1-score   